# Module 05: Basic Chart Types with Matplotlib
**CHIRPS Rainfall Data – Ethiopia**

In this module you will learn the fundamental chart types available in Matplotlib.
All examples use real CHIRPS (Climate Hazards Group InfraRed Precipitation with Station)
precipitation data for Ethiopia at 0.05° resolution, January 1981 – December 2022.




In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

# Load CHIRPS dataset
ds = xr.open_dataset(r'../chirps_1981_2022.nc', engine='netcdf4')
print(f"Dataset shape: {ds.dims}")
print(f"Time range: {ds.time.values[0]} to {ds.time.values[-1]}")
print(f"Mean precipitation: {ds.precip.mean().values:.2f} mm/month")



In [ ]:
# Extract time series at Addis Ababa (9.025N, 38.725E)
lat_idx, lon_idx = 180, 174
ts = ds.precip.isel(latitude=lat_idx, longitude=lon_idx)
ts_df = pd.DataFrame({'precip': ts.values}, index=pd.DatetimeIndex(ts.time.values))
print(f"Location: lat={ds.latitude[lat_idx].values:.3f}, lon={ds.longitude[lon_idx].values:.3f}")
print(f"Record length: {len(ts)} months")
print(f"Annual mean: {ts_df.resample('YE').mean().mean().values[0]:.1f} mm/month")



---
## 5.1 Line Plot – Rainfall Time Series

The **line plot** is the most basic chart type. It connects data points with straight line segments
and is ideal for showing trends over time.



In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(ts_df.index, ts_df['precip'], linewidth=0.6, color='steelblue')
ax.set_title('CHIRPS Monthly Precipitation at Addis Ababa (9.025N, 38.725E)', fontsize=13)
ax.set_xlabel('Year')
ax.set_ylabel('Precipitation (mm/month)')
ax.set_xlim(ts_df.index[0], ts_df.index[-1])
plt.tight_layout()
plt.show()



In [ ]:
# 5-year subset for clarity
fig, ax = plt.subplots(figsize=(12, 4))
sub = ts_df.loc['2000':'2004']
ax.plot(sub.index, sub['precip'], marker='o', markersize=3, linewidth=1, color='coral')
ax.set_title('CHIRPS Precipitation 2000–2004 at Addis Ababa')
ax.set_xlabel('Date'); ax.set_ylabel('Precipitation (mm/month)')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()



---
## 5.2 Scatter Plot – Lag Correlation

Scatter plots show the relationship between two variables. Here we examine
**autocorrelation**: does this month's rainfall predict next month's?



In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
precip = ts_df['precip'].values
ax.scatter(precip[:-1], precip[1:], s=8, alpha=0.4, color='darkgreen')
ax.set_title('Lag-1 Autocorrelation – Addis Ababa Rainfall')
ax.set_xlabel('Precipitation month t (mm)')
ax.set_ylabel('Precipitation month t+1 (mm)')

# 1:1 line
lims = [0, max(precip)]
ax.plot(lims, lims, 'r--', linewidth=0.8, label='1:1')
ax.legend()
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_aspect('equal')
plt.tight_layout(); plt.show()

corr = np.corrcoef(precip[:-1], precip[1:])[0, 1]
print(f"Lag-1 Pearson correlation: {corr:.3f}")



---
## 5.3 Bar Chart – Monthly Climatology

Bar charts use vertical bars to represent quantities. The **climatology** (mean annual cycle)
is a classic application: average rainfall for each calendar month.



In [ ]:
monthly_mean = ts_df.groupby(ts_df.index.month).mean()
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(months, monthly_mean['precip'], color='royalblue', edgecolor='navy', linewidth=0.8)
ax.set_title('Monthly Rainfall Climatology – Addis Ababa (1981–2022)')
ax.set_xlabel('Month'); ax.set_ylabel('Mean Precipitation (mm/month)')
ax.set_ylim(0, monthly_mean['precip'].max() * 1.15)

# Add value labels
for bar, val in zip(bars, monthly_mean['precip']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'{val:.0f}',
            ha='center', va='bottom', fontsize=8)
plt.tight_layout(); plt.show()



---
## 5.4 Horizontal Bar Chart – Regional Comparison

Horizontal bars are useful when category labels are long or when comparing many items.



In [ ]:
# Extract mean rainfall for several Ethiopian regions
regions = {
    'Addis Ababa':  (180, 174),
    'Gondar':       (252, 150),
    'Mekelle':      (270, 190),
    'Dire Dawa':    (192, 237),
    'Jimma':        (  8, 144),
    'Bahir Dar':    (235, 140),
}
means = {}
for name, (li, lo) in regions.items():
    means[name] = float(ds.precip.isel(latitude=li, longitude=lo).mean().values)

fig, ax = plt.subplots(figsize=(9, 5))
names_list = list(means.keys())
vals_list  = list(means.values())
bars = ax.barh(names_list, vals_list, color='teal', edgecolor='darkcyan')
ax.set_title('Mean Monthly Rainfall – Ethiopian Regions (1981–2022)')
ax.set_xlabel('Precipitation (mm/month)')
for bar, v in zip(bars, vals_list):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, f'{v:.1f}',
            va='center', fontsize=9)
ax.set_xlim(0, max(vals_list) * 1.2)
plt.tight_layout(); plt.show()



---
## 5.5 Histogram – Rainfall Distribution

Histograms show the **frequency distribution** of a continuous variable.
They reveal the shape, spread, and central tendency of the data.



In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(ts_df['precip'], bins=40, color='mediumseagreen', edgecolor='white', linewidth=0.6)
ax.set_title('Distribution of Monthly Rainfall – Addis Ababa')
ax.set_xlabel('Precipitation (mm/month)'); ax.set_ylabel('Frequency (months)')
ax.axvline(ts_df['precip'].mean(), color='darkred', linestyle='--', linewidth=1.5,
           label=f"Mean = {ts_df['precip'].mean():.1f} mm")
ax.axvline(ts_df['precip'].median(), color='orange', linestyle=':', linewidth=1.5,
           label=f"Median = {ts_df['precip'].median():.0f} mm")
ax.legend()
plt.tight_layout(); plt.show()



---
## 5.6 Pie Chart – Seasonal Rainfall Proportion

Pie charts show proportions of a whole. Despite their popularity, they should be used sparingly.
Here we show the seasonal contribution to total annual rainfall.



In [ ]:
# Define seasons: DJF, MAM, JJA, SON
ts_df['season'] = ts_df.index.month.map({12:'DJF',1:'DJF',2:'DJF',
                                          3:'MAM',4:'MAM',5:'MAM',
                                          6:'JJA',7:'JJA',8:'JJA',
                                          9:'SON',10:'SON',11:'SON'})
seasonal = ts_df.groupby('season')['precip'].sum()
# Reorder
seasonal = seasonal[['DJF','MAM','JJA','SON']]
season_labels = ['DJF (Winter)', 'MAM (Spring)', 'JJA (Summer)', 'SON (Autumn)']

fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    seasonal.values, labels=season_labels, autopct='%1.1f%%',
    colors=['cornflowerblue', 'lightgreen', 'gold', 'lightcoral'],
    startangle=90, explode=(0, 0, 0.05, 0),
    textprops={'fontsize': 11}
)
ax.set_title('Seasonal Rainfall Proportion – Addis Ababa', fontsize=13, pad=20)
plt.tight_layout(); plt.show()



---
## 5.7 Stem Plot

Stem plots (or *lollipop charts*) draw lines from a baseline to each data point.
They work well for discrete data or when you want to emphasise individual values.



In [ ]:
# Last 24 months of the record
last24 = ts_df.iloc[-24:]
fig, ax = plt.subplots(figsize=(12, 4))
markerline, stemlines, baseline = ax.stem(
    last24.index, last24['precip'],
    linefmt='-b', markerfmt='ob', basefmt='-k', use_line_collection=True
)
ax.set_title('Stem Plot – Last 24 Months at Addis Ababa')
ax.set_ylabel('Precipitation (mm/month)')
ax.set_xlabel('Date')
plt.tight_layout(); plt.show()



---
## 5.8 Step Plot

Step plots show changes as discrete jumps, which is useful for highlighting
when thresholds are crossed or for cumulative data.



In [ ]:
cumulative = ts_df['precip'].cumsum() / 1000  # in metres
fig, ax = plt.subplots(figsize=(14, 4))
ax.step(ts_df.index, cumulative, where='mid', linewidth=1.2, color='darkorange')
ax.set_title('Cumulative Precipitation – Addis Ababa (Step Plot)')
ax.set_xlabel('Year'); ax.set_ylabel('Cumulative Precip (m)')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Total cumulative rainfall: {cumulative.iloc[-1]:.2f} m over {len(ts_df)} months")



---
## Exercises – Module 05

1. **Line plot**: Extract the time series for **Gondar** (lat_idx=252, lon_idx=150) and create a
   line plot of monthly rainfall from 2010 to 2020. Add markers for months exceeding 200 mm.

2. **Scatter plot**: Create a scatter plot of **Mekelle** rainfall vs **Addis Ababa** rainfall
   (same time steps). Compute and display the correlation coefficient.

3. **Bar chart**: Compute the monthly climatology for **Dire Dawa** and plot it as a bar chart.
   Which month is the wettest?

4. **Horizontal bar chart**: Extract mean rainfall for 8 Ethiopian cities and create a horizontal
   bar chart sorted from wettest to driest.

5. **Histogram**: Plot side-by-side histograms of rainfall for the **short rainy season** (MAM)
   and **long rainy season** (JJA) in Addis Ababa. Use 20 bins, alpha=0.6.

6. **Pie chart**: Create a pie chart showing the proportion of rainfall contributed by each
   calendar month at a location of your choice.

7. **Stem plot**: Create a stem plot of the 12 monthly climatology values.

8. **Step plot**: Create a step plot showing the cumulative rainfall for the **year 1997**
   (the El Niño year) and compare its shape to a normal year (e.g. 2000).

### Mini-Project: Rainfall Regime Classification

Create a 2×3 figure that characterises the rainfall regime at **Addis Ababa**:
- Top-left: Line plot of the full time series
- Top-middle: Bar chart of monthly climatology
- Top-right: Histogram of monthly rainfall
- Middle-left: Seasonal pie chart
- Middle-middle: Scatter of Oct–Dec vs Mar–May totals (interannual variability)
- Middle-right: Step plot of 1997 cumulative rainfall

Use `plt.subplots(nrows=2, ncols=3, figsize=(16, 10))`.



In [ ]:
# Mini-Project starter
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Line plot
axes[0,0].plot(ts_df.index, ts_df['precip'], linewidth=0.5, color='steelblue')
axes[0,0].set_title('Full Time Series'); axes[0,0].set_ylabel('mm/month')

# 2. Bar chart – climatology
monthly = ts_df.groupby(ts_df.index.month)['precip'].mean()
axes[0,1].bar(months, monthly, color='royalblue', edgecolor='navy')
axes[0,1].set_title('Monthly Climatology')

# 3. Histogram
axes[0,2].hist(ts_df['precip'], bins=40, color='mediumseagreen', edgecolor='white')
axes[0,2].axvline(ts_df['precip'].mean(), color='red', ls='--')
axes[0,2].set_title('Rainfall Distribution'); axes[0,2].set_xlabel('mm/month')

# 4. Seasonal pie
seasonal = ts_df.groupby('season')['precip'].sum()
seasonal = seasonal[['DJF','MAM','JJA','SON']]
axes[1,0].pie(seasonal, labels=seasonal.index, autopct='%1.0f%%')
axes[1,0].set_title('Seasonal Proportions')

# 5. Scatter: Oct-Dec vs Mar-May
oct_dec = ts_df[ts_df.index.month.isin([10,11,12])].groupby(ts_df[ts_df.index.month.isin([10,11,12])].index.year)['precip'].sum()
mar_may = ts_df[ts_df.index.month.isin([3,4,5])].groupby(ts_df[ts_df.index.month.isin([3,4,5])].index.year)['precip'].sum()
axes[1,1].scatter(oct_dec, mar_may, s=20, alpha=0.6, color='darkgreen')
axes[1,1].set_xlabel('OND total (mm)'); axes[1,1].set_ylabel('MAM total (mm)')
axes[1,1].set_title('OND vs MAM Totals')

# 6. Step plot – 1997 cumulative
year97 = ts_df.loc['1997']
cum97 = year97['precip'].cumsum()
axes[1,2].step(cum97.index, cum97, where='mid', color='darkorange')
axes[1,2].set_title('1997 Cumulative'); axes[1,2].set_ylabel('mm')

plt.tight_layout()
plt.show()

